# 实验：vLLM BaseDemand-only Deep Research Agent

这份 notebook 用于在云端运行只包含 BaseDemand 三部分的 agent：多轮检索 loop、上下文管理、prompt 设计。

它调用 `agent/base_demand_agent.py`，只使用课程基础工具 `search` 和 `get_document`，不包含加分赛道工具、多 Agent 或微调逻辑。

## 0. 环境与服务说明

运行前假设你已经在云端启动 vLLM OpenAI-compatible 服务，并且 BM25 索引路径与数据集路径和原 notebook 保持一致。

本 notebook 会依次完成：

1. 连接 vLLM 服务
2. 初始化 BaseDemand-only agent
3. 单条样例 smoke test
4. 批量生成 `submission.jsonl`
5. 调用 `agent.eval` 自动评测

In [ ]:
!python --version
!pip install -r agent/requirements.txt

## 1. vLLM 服务配置

如果你使用课程推荐的 Qwen3-8B 服务，通常模型名为 `qwen_auto`，服务地址为 `http://127.0.0.1:8000/v1`。

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

VLLM_BASE_URL = 'http://127.0.0.1:8000/v1'
MODEL_NAME = 'qwen_auto'
# MODEL_NAME = 'pangu_auto'
API_KEY = 'dummy'

## 2. BM25 索引路径

如果索引已经构建过，可以跳过构建索引的单元格，只保留路径变量。

In [ ]:
corpus_path = str('browsecomp-plus-corpus')
bm25_index_path = str('indexes/browsecomp_plus_bm25.sqlite')

# 首次在服务器上执行；已经建过索引就跳过本单元格。
!python -m agent.build_bm25_index --corpus-path ./browsecomp-plus-corpus --index-path ./indexes/browsecomp_plus_bm25.sqlite --overwrite

## 3. 初始化 BaseDemand-only agent

In [ ]:
import importlib
from agent.vllm_client import VLLMClient
from agent.tools import build_searcher, get_agent_tool_specs_and_registry
from agent.dataset_utils import load_jsonl
import agent.base_demand_agent as base_demand_agent

base_demand_agent = importlib.reload(base_demand_agent)
run_base_demand_agent = base_demand_agent.run_base_demand_agent
generate_submission = base_demand_agent.generate_submission

hard50_path = str(project_root / 'browsecomp_plus_hard50.jsonl')
submission_path = str(project_root / 'runs' / 'base_demand_only_submission.jsonl')
eval_output_path = str(project_root / 'runs' / 'base_demand_only_eval_results.jsonl')

client = VLLMClient(base_url=VLLM_BASE_URL, api_key=API_KEY)
searcher = build_searcher(index_path=bm25_index_path)
tool_specs, tool_registry = get_agent_tool_specs_and_registry(searcher=searcher, k=5, snippet_max_chars=1200)
print('search_type:', searcher.search_type)
print('submission_path:', submission_path)
print('eval_output_path:', eval_output_path)

## 4. BaseDemand 参数

这版只使用基础工具 `search` 和 `get_document`。为了控制成本，默认决策输出 token 比多 Agent 定版更小。

In [ ]:
TOP_K = 5
MAX_ROUNDS = 7
DECISION_MAX_TOKENS = 384
ANSWER_MAX_TOKENS = 512
SEARCH_SNIPPET_MAX_CHARS = 1200
TOOL_CONTENT_MAX_CHARS = 4000

## 5. 单条样例 smoke test

先跑 1 条，确认轨迹里只出现 `search` 和 `get_document`。

In [ ]:
rows = load_jsonl(hard50_path, limit=1)
demo_row = rows[0]

demo = run_base_demand_agent(
    question=demo_row['query'],
    client=client,
    model_name=MODEL_NAME,
    tool_registry=tool_registry,
    max_rounds=MAX_ROUNDS,
    decision_max_tokens=DECISION_MAX_TOKENS,
    answer_max_tokens=ANSWER_MAX_TOKENS,
    tool_content_max_chars=TOOL_CONTENT_MAX_CHARS,
)

print('query_id:', demo_row['query_id'])
print('gold_answer:', demo_row['answer'])
print('predicted_answer:', demo['predicted_answer'])
print('messages:', len(demo['messages']))
print('\nmessage roles and tool names:')
for msg in demo['messages']:
    if msg['role'] == 'assistant' and 'tool_calls' in msg:
        names = [call['function']['name'] for call in msg['tool_calls']]
        print('-', msg['role'], names)
    else:
        print('-', msg['role'], list(msg.keys()))

## 6. 批量生成 submission.jsonl

正式跑 `hard50` 时使用 `limit=50`。如果只想快速试跑，可临时改成 `limit=5` 或 `limit=10`。

In [ ]:
rows = load_jsonl(hard50_path, limit=50)

records = generate_submission(
    dataset_rows=rows,
    index_path=bm25_index_path,
    base_url=VLLM_BASE_URL,
    model_name=MODEL_NAME,
    output_path=submission_path,
    api_key=API_KEY,
    top_k=TOP_K,
    search_snippet_max_chars=SEARCH_SNIPPET_MAX_CHARS,
    tool_content_max_chars=TOOL_CONTENT_MAX_CHARS,
    max_rounds=MAX_ROUNDS,
    decision_max_tokens=DECISION_MAX_TOKENS,
    answer_max_tokens=ANSWER_MAX_TOKENS,
)

print('\nSaved to:', submission_path)
print('num_records:', len(records))

sample = records[0]
print('\n--- first trajectory sample ---')
print('query_id:', sample['query_id'])
print('status:', sample['status'])
print('predicted_answer:', sample['predicted_answer'])
print('messages:', len(sample['messages']))

## 7. 自动评测

`agent.eval` 仍然会调用同一个 vLLM 服务，用模型判断预测答案与标准答案是否一致。

In [ ]:
from agent.eval import run_evaluation

summary, details = run_evaluation(
    submission_path=submission_path,
    dataset_path=hard50_path,
    model_name=MODEL_NAME,
    base_url=VLLM_BASE_URL,
    api_key=API_KEY,
    output_path=eval_output_path,
    temperature=0.0,
    max_tokens=256,
    verbose=True,
)

print('\nSummary:')
print(summary)

## 8. 查看评测结果样例

In [ ]:
import json
from pathlib import Path

eval_lines = [json.loads(line) for line in Path(eval_output_path).read_text(encoding='utf-8').splitlines() if line.strip()]
print('summary line:')
print(eval_lines[0])
if len(eval_lines) > 1:
    print('\nfirst detail line:')
    print(json.dumps(eval_lines[1], ensure_ascii=False, indent=2)[:3000])

## 9. 快速检查 BaseDemand 约束

这个检查只确认生成轨迹中的工具名是否限定为 `search` 和 `get_document`。

In [ ]:
allowed_tools = {'search', 'get_document'}
used_tools = set()
for record in records:
    for msg in record.get('messages', []):
        for call in msg.get('tool_calls', []) if isinstance(msg, dict) else []:
            used_tools.add(call.get('function', {}).get('name', ''))

print('used_tools:', sorted(used_tools))
print('base_demand_only:', used_tools <= allowed_tools)